# 1장. 교육 전체 개요와 환경 준비 — CAE/Python 입문자용

## 개요

**학습 목표:** 전체 교육 흐름 파악 / 실습환경 준비와 동작 확인

| 읽을 순서 | 알아볼 내용 |
|---|---|
| 1. 교육 전체 개요 | CAE 데이터 학습/Omniverse 추론의 흐름, 교육용 예제, Day 1/Day 2의 구성 |
| 2. 실습 도구와 실행환경 | Python/Notebook/PhysicsNeMo/Omniverse의 역할 |
| 3. 실습환경 준비 | Ansys Python Manager로 실습환경 준비 |
| 4. 폴더와 Notebook | 교육 파일 복사, 가상환경 생성, 첫 코드 실행 |
| 5. 실행환경 확인 | GPU/패키지/해석 파일/Kit이 실제로 동작하는지 확인 |
| 6. 점검 결과 저장 | 실행환경 점검 결과를 JSON 파일로 저장 |


<div class="emphasis-key">강조 안내: <mark class="lesson-mark" data-kind="concept" title="핵심 개념" style="background-color:#fff1ad;color:#242424">핵심 개념</mark> <mark class="lesson-mark" data-kind="action" title="실습 행동" style="background-color:#d6ebff;color:#242424">실습 행동</mark> <mark class="lesson-mark" data-kind="caution" title="주의사항" style="background-color:#f8dce4;color:#242424">주의사항</mark></div>


## 1. 교육 전체 개요와 흐름


### 1.1 CAE 데이터 학습과 Omniverse 추론은 어떻게 이어지나요?


- **CAE:** 형상/재료/하중/고정 조건을 바탕으로 제품의 성능과 거동을 계산하는 공학 활동
- **AI 대리모델:** 해석 데이터의 입력/결과 관계를 학습해 결과를 예측하는 도구
- **케이스(case):** 한 조건의 해석 자료 묶음 / 같은 이름으로 두께/하중 조건과 후크 전체 해석 결과 연결
- **학습:** 예측과 정답의 차이가 줄도록 모델 내부 숫자(가중치) 조정
- **추론:** 학습한 모델에 입력을 넣어 예측 결과 계산
- **이 교육의 정답:** ANSYS 구조해석 결과. 실제 시험 측정값과 구분

| 용어 | 쉬운 설명 |
|---|---|
| 하중 | 물체에 가하는 힘. 이번 예제에서 크기의 단위는 N(뉴턴) |
| 변위 | 원래 위치에서 얼마나, 어느 방향으로 이동했는지. 단위는 mm |
| 응력 | 힘을 받은 물체 내부의 힘 전달 상태를 나타내는 값. 단위는 MPa |
| 해석 조건 | 계산할 때 정해 놓은 형상/재료/하중/고정 방법 등 |

**결과를 읽을 때 구분할 점**

- 하중과 응력은 서로 다른 물리량. 같은 하중이라도 형상/두께에 따라 응력이 달라짐
- 최대 변위 위치와 최대응력 위치는 서로 다를 수 있음
- 예측할 응력: 노드별 **von Mises 응력(등가응력)** 값 하나
- **API:** 프로그램끼리 요청과 결과를 주고받는 방법을 정한 인터페이스
- **추론 서버:** API 요청 처리와 예측 결과 반환. 새 조건은 모델 계산, 유효한 동일 조건 결과는 재사용
- **Omniverse:** 조건 전달 → 결과의 색상/최대점/수치 표시 → 조건/모델 출처 확인


```text
준비된 CAE 데이터 → 학습 자료 구성 → AI 모델 학습/저장
  → 추론/정답 비교 → API 연결 → Omniverse에서 예측 요청/결과 확인
```


### 1.2 CAE 데이터로 무엇을 학습하고 무엇을 예측하나요?


**학습 자료의 기본 구성:** CAE 형상/조건/결과를 입력과 정답으로 연결

입력 변수와 예측 대상은 문제의 목적에 맞게 선정.

| 구분 | 담는 정보 | 구조해석 데이터에서의 의미 |
|---|---|---|
| 형상/위치 정보 | 계산 대상의 모양과 결과를 구할 위치 | 메시, 노드 번호, X/Y/Z 좌표 |
| 설계/해석 조건 | 결과에 영향을 주는 변수와 고정된 조건 | 치수, 재료, 하중, 고정 방법 등 |
| 학습 정답 | CAE가 해당 조건에서 계산한 결과 | 위치별 응력/변위 등 |
| AI 예측 | 학습 모델이 입력을 받아 계산한 결과 | 같은 위치와 조건에 대한 예측값 |
| Omniverse 표시 | 예측값과 그 값의 출처를 장면에 연결 | 결과 분포, 최대값/위치, 조건/모델 정보 |

- **요소:** 계산을 위해 나눈 작은 조각
- **노드:** 요소를 정의하는 점
- **메시:** 노드와 요소로 구성한 계산용 격자
- **자료 대응:** 같은 노드의 좌표와 정답을 한 쌍으로 연결. 번호/순서가 어긋나면 다른 위치의 값을 학습/표시하게 됨
- **바꾸는 조건:** 두께/총하중
- **유지하는 조건:** 재료/하중 방향/고정 방법
- **후크 모델 입력:** 노드의 X/Y/Z 좌표 + 두께/총하중 = 숫자 5개
- **후크 모델 출력:** 같은 노드의 등가응력 1개 + X/Y/Z 변위 3개
- **두께 변경:** 형상에 맞게 노드 좌표도 변경

| 자료의 역할 | 사용하는 이유 |
|---|---|
| train — 학습 자료 | 입력과 정답의 관계를 배우도록 모델 조정 |
| validation — 검증 자료 | 학습 중 상태를 확인하고 저장 모델 선택 |
| test — 시험 자료 | 학습과 모델 선택에 사용하지 않은 조건에서 성능 확인 |

- **정답이 있는 조건:** 같은 조건의 정답과 예측을 비교해 오차 평가
- **정답이 없는 조건:** 입력 변화에 따른 모델 반응 관찰
- **2~4장:** 구체적인 자료 구성과 모델 입출력 확인


### 1.3 CAE 데이터에서 Omniverse 추론까지의 작업 흐름


**DOE 기반 준비 → 학습 자료 구성 → 모델 학습 → 추론/평가 → API/Omniverse 연결**

- **DOE(실험계획법):** 변수/범위와 검토할 조건 조합을 계획하는 방법
- **자료 준비 목적:** 특정 조건에 치우치지 않는 데이터 구성
- **이번 교육의 시작점:** DOE와 해석이 완료된 자료

| 단계 | 수행하는 작업 | 다음 단계에 전달할 것 |
|---|---|---|
| 1. DOE 기반 데이터 준비 | 변수/범위/조건 조합을 계획해 CAE 결과 준비. 수업에서는 준비된 조건표/형상/결과의 연결 확인 | DOE 조건표/CAE 결과/단위/출처 |
| 2. 학습 자료 구성 | 입력과 정답 연결, 학습/검증/시험 분할, 숫자 범위 정규화 | Dataset/분할/정규화 통계 |
| 3. 모델 학습과 저장 | 예측이 정답에 가까워지도록 학습하고 검증 기준으로 모델 선택 | 모델 파일/설정/학습 기록 |
| 4. 추론과 평가 | 저장 모델을 불러와 예측하고 같은 조건의 해석 정답과 비교 | 예측 결과/오차/사용 모델 정보 |
| 5. API 서비스 연결 | 외부 요청을 받아 같은 모델로 계산하고 결과 전달 | 예측 요청/응답 통로 |
| 6. Omniverse 연결 | 조건을 선택해 추론을 요청하고 결과를 장면과 수치로 표시 | 결과 화면/조건/모델 출처 |

![DOE 기반 CAE 데이터 준비와 Omniverse 연계 추론의 흐름](../assets/01_beginner_cae_omniverse_flow.svg)

**단계 사이의 자료 연결**

1. 조건표의 케이스 이름 ↔ 해석 결과 파일
2. 학습 모델 ↔ 학습 자료로 구한 정규화 기준
3. 평가한 모델/기준 ↔ 추론 서버
4. 응답 노드 순서/단위 ↔ Omniverse의 같은 위치

**아래 표:** 이후 장에서 사용할 자료의 연결 안내. 지금 모두 암기할 대상은 아님.

**사용 단계:** 위 작업 흐름 표의 1~6단계와 같은 번호

| 자료/형식 | 역할과 만드는 방법 | 사용 단계 |
|---|---|---|
| CSV | DOE의 두께/하중 조건이나 평가 결과를 행과 열로 정리한 표 | 1. DOE 기반 데이터 준비 / 4. 추론과 평가 |
| RST | ANSYS 구조해석으로 얻은 메시와 위치별 응력/변위를 담은 결과 파일 | 1. DOE 기반 데이터 준비: 케이스별 결과 확보와 변환 |
| VTU | ANSYS 결과 RST → PyDPF 변환 → 메시/좌표/응력/변위를 담은 파일 | 1. DOE 기반 데이터 준비에서 생성 → 2. 학습 자료 구성에서 읽기 |
| PyTorch Dataset | CSV 조건 + VTU 좌표/응력/변위로 구성. 한 케이스의 입력/정답을 꺼내 제공 | 2. 학습 자료 구성에서 생성 → 3. 모델 학습 / 4. 추론과 평가에 활용 |
| 정규화 통계 | 학습 자료에서 계산한 평균/표준편차. 입력/정답의 범위를 조정하고 예측을 원래 단위로 복원하는 기준 | 2. 학습 자료 구성에서 계산 → 3~5단계에서 같은 기준 사용 |
| 체크포인트 | 학습한 모델의 가중치와 설정 등을 저장해 다시 사용하는 파일 | 3. 모델 학습과 저장에서 생성 → 4. 추론과 평가 / 5. API 서비스 연결에서 불러오기 |
| API 응답 | 추론 서버가 계산한 예측값과 사용 조건/모델 정보를 전달하는 자료 | 5. API 서비스 연결에서 반환 → 6. Omniverse 연결에서 표시 |
| USD 장면 | 형상과 표시 속성/결과 출처를 담는 3D 장면 | 6. Omniverse 연결에서 구성하고 저장 |

**연결 시 확인할 항목:** 모델/데이터/정규화 통계/단위

좌표의 mm와 m를 혼동하면 위치의 의미가 달라짐. 화면 표시뿐 아니라 데이터의 대응 관계도 확인 필요.


### 1.4 Day 1/Day 2에는 무엇을 하나요?


**Day 1: 해석 데이터 이해와 온도 예제의 모델 학습**

**예제 연결:** 4장 공통 온도 문제로 모델 차이 비교 → 5장 후크 응력/변위 학습에 적용

| 장 | 배울 내용 | 직접 해 볼 일 |
|---|---|---|
| [1. 개요와 환경 준비](01_environment_setup_beginner.ipynb) | 전체 흐름과 도구의 역할 | 프로그램을 준비하고 기본 실행 결과 확인 |
| [2. CAE 학습 데이터 준비와 ANSYS 결과 변환](02_rst_to_vtu_beginner.ipynb) | DOE 조건 선정, Workbench를 통한 ANSYS 해석 데이터 준비, 결과 구조 | 준비된 RST를 PyDPF로 읽어 VTU로 정리 |
| [3. CAE 학습 데이터 구성과 전처리](03_dataset_beginner.ipynb) | 입력/정답/학습/검증/시험 자료 | 데이터를 나누고 숫자의 범위를 조정 |
| [4. PhysicsNeMo의 AI 모델 이해와 사용](04_models_beginner.ipynb) | GNN/Neural Operator와 FNO/Transolver의 개념 | 공통 온도 자료로 세 모델을 학습하고 예측/오차 확인 |

**Day 2: 후크 모델 학습/평가와 화면 연결**

| 장 | 배울 내용 | 직접 해 볼 일 |
|---|---|---|
| [5. AI 예측 모델로 후크의 응력/변위 학습하기](05_training_beginner.ipynb) | 학습/검증과 모델 선택, 학습 횟수/노드 수의 영향 | 100/1,000에폭과 노드 수 비교를 수행하고 모델/학습 기록 저장 |
| [6. AI 예측 모델로 후크의 응력/변위를 예측하고 평가하기](06_inference_evaluation_beginner.ipynb) | 남겨 둔 시험 케이스에서 오차 평가 | 5장의 같은 모델을 불러와 ANSYS 정답과 그림/MAE/RMSE/최대값/위치 비교 |
| [7. Omniverse 연결을 위한 AI 추론 서버 준비](07_regions_load_study_beginner.ipynb) | 두께 모핑과 추론 서버, 요청/응답의 역할 | FastAPI/Uvicorn 서버를 실행하고 두께/하중 요청과 결과 전달 확인 |
| [8. Omniverse에서 후크의 두께/하중에 따른 응력/변위 확인하기](08_omniverse_visualization_beginner.ipynb) | 슬라이더 요청과 형상/응력/변위 크기 표시 | 두께/하중을 바꾸어 예측을 요청하고 응답 상태/단위/최대값 확인 |

- **장별 진행:** 설명 → 코드 실행 → 결과 확인
- **모델 연결:** 5장에서 본인이 학습한 기본 모델을 6~8장에 이어 사용
- **강사 제공 참고 모델:** 별도 선택 및 사용 여부 표시
- **시간 운영:** 설치/계산 시간에 강사 설명/조작/질문/복습 시간을 더해 조정


## 2. 실습 도구와 실행환경의 역할


### 2.1 Python/패키지/가상환경/Notebook


**Python:** 컴퓨터가 수행할 작업 순서를 작성하는 언어

**Python 설치:** 작성한 코드를 실행할 프로그램 준비

| 용어 | 쉬운 설명 | 이번 교육에서의 사용 |
|---|---|---|
| 코드 | 컴퓨터가 수행할 일을 적은 문장 | 파일 읽기, 모델 계산, 결과 확인 |
| 패키지 | 다른 사람이 만들어 놓은 기능 묶음 | NumPy로 숫자 처리, PyVista로 해석 데이터 읽기 |
| pip | Python 패키지를 설치하는 도구 | 필요한 기능을 가상환경에 추가 |
| 가상환경 | 프로젝트별로 Python 패키지를 따로 관리하는 공간 | 교육에 필요한 패키지들을 한곳에서 관리 |
| Ansys Python Manager | Python 설치와 가상환경 관리를 도와주는 프로그램 | 버튼과 메뉴로 Python/Notebook 준비 |
| Jupyter Notebook | 설명/코드/실행 결과를 한 문서에서 보는 도구 | 지금 읽는 `.ipynb` 교재 실행 |
| 커널(kernel) | Notebook의 Python 코드를 실제로 실행하는 프로그램 | 셀 사이의 변수와 실행 상태를 기억 |
| Console(콘솔) | 명령을 입력하고 실행 결과를 보는 창 | 패키지 설치, GPU 확인, Kit 실행 |

- **가상 장비:** 실습에 사용하는 Windows 컴퓨터 전체
- **Python 가상환경:** 프로젝트별 패키지를 따로 관리하는 공간
- **작업 폴더:** 교재/데이터/결과 보관 위치
- **환경 일치 조건:** 패키지를 설치한 Python과 Notebook 커널의 Python이 같아야 함. Console/Notebook의 실제 Python 경로를 비교해 확인


### 2.2 PyTorch와 PhysicsNeMo


- **PyTorch:** 숫자 계산/손실 계산/가중치 조정 기능 제공
- **PhysicsNeMo:** PyTorch 기반 과학/공학 AI 프레임워크
- **프레임워크:** 기능과 구조를 함께 제공하는 개발 기반
- **4장:** 같은 온도 자료로 GNN/FNO/Transolver 학습/비교. FNO는 Neural Operator 계열
- **5장:** Transolver로 후크 응력/변위 학습
- **학습 방식:** 정답을 보고 배우는 **지도학습**. PhysicsNeMo 사용만으로 물리 방정식을 자동 학습하는 것은 아님


### 2.3 Omniverse/Kit/USD와 Python 실행환경


- **Omniverse:** 3D 응용 프로그램과 작업 흐름을 구성하는 플랫폼/도구 생태계
- **Kit:** 예제 형상과 추론 결과를 표시할 프로그램의 실행 기반
- **USD(OpenUSD):** 3D 형상/재질/장면 정보를 표현/구성하는 체계

| 프로그램 쪽 | 하는 일 |
|---|---|
| Manager로 준비한 Python | 데이터를 읽고, 모델을 학습하고, 예측값을 계산 |
| Kit에 들어 있는 Python | 예제 형상과 추론 결과의 색상/수치를 3D 화면에 표시 |
| 두 프로그램을 연결하는 API | Kit의 예측 요청을 전달하고 계산 결과를 돌려줌 |

**API 요청 항목:** 두께/총하중 두 값

**모핑:** 기준 형상의 노드 좌표를 바꾸어 요청 두께의 형상 구성

예: 새 조건 9 mm/300 N 요청 → 서버에서 두께 9 mm로 모핑 → 전체 노드 X/Y/Z와 두 조건으로 추론 → Kit 표시. 유효한 동일 조건 결과는 재사용.

![학습용 Python과 Kit의 연결](../assets/01_revised_process_flow.svg)

- **후크 실습 Notebook/API 서버:** 같은 교육용 Python 가상환경에서 각각 실행
- **<mark class="lesson-mark" data-kind="concept" title="핵심 개념" style="background-color:#fff1ad;color:#242424">4장 모델 예제</mark>:** 별도 전용 커널 사용
- **Kit:** 자체 Python 사용
- **`localhost:8098`:** 같은 컴퓨터에서 실행 중인 API 서버의 주소
- **연결 순서:** 7장 서버 준비 → 8장 Omniverse 화면 연결

참고: [PhysicsNeMo 개요](https://docs.nvidia.com/physicsnemo/latest/index.html), [PyTorch 모델과의 연결/저장 기능](https://docs.nvidia.com/physicsnemo/latest/physicsnemo/api/models/modules.html), [Kit 개요](https://docs.omniverse.nvidia.com/kit/docs/kit-manual/latest/guide/kit_overview.html), [OpenUSD 소개](https://openusd.org/release/intro.html)


## 3. Ansys Python Manager로 실습환경 준비하기


### 3.1 Ansys Python Manager 실행


**실행 장소: Windows 바탕화면/시작 메뉴**

1. 제공된 Ansys Python Manager 설치 파일 실행. 직접 다운로드할 경우 공식 Releases의 강사 지정 Windows 버전 선택
2. 시작 메뉴 → **Ansys Python Manager** 실행
3. 설치 버전 기록. 버전에 따라 버튼 이름이 다를 수 있음


### 3.2 개인 폴더를 만들고 Ansys Python Manager에 알려주기


**경로(path):** 파일/폴더의 위치를 나타내는 주소

**개인 폴더 예시:** `X:\student01` — `student01`은 본인의 영문 폴더 이름으로 변경

1. 파일 탐색기 → **Temp (X:)** 열기
2. 개인 폴더 생성. 예: `X:\student01`
3. Manager → **File → Configuration** 열기. 버전에 따라 **Configure**로 표시
4. **Default virtual environment creation path** → 개인 폴더 경로 입력
5. **Locations to search for virtual environments** → 같은 경로 입력 후 **Add** 선택
6. **Save**로 설정 저장

- **첫 번째 설정:** 가상환경 생성 위치
- **두 번째 설정:** 가상환경 검색 위치
- **X: 드라이브가 없는 경우:** 강사 지정 드라이브 사용


### 3.3 Python 3.12 설치하기


1. **Install Python** 탭 선택
2. **Installation type: Standard**, **Python version: 3.12 계열** 선택
3. **Install** 선택 후 설치 완료까지 대기
4. 실제 설치 버전/경로 기록

**기존 가상 장비의 점검 버전:** Python 3.12.10

교육 지정 3.12 계열로 설치한 뒤, 다음 단계에서 해당 Python으로 가상환경 생성.

참고: [공식 Releases](https://github.com/ansys/python-installer-qt-gui/releases), [Manager 경로 설정 안내](https://installer.docs.pyansys.com/version/stable/installer.html), [Manager Python 설치 안내](https://installer.docs.pyansys.com/version/stable/user_guide.html#install-python-tab)


## 4. 폴더를 준비하고 첫 Notebook 열기


### 4.1 교육 파일을 담을 작업 폴더 만들기


**실행 장소: 파일 탐색기**

1. 개인 폴더 안에 **workshop** 생성. 예: `X:\student01\workshop`
2. 배포 압축 해제 → **education 폴더 전체**를 workshop에 복사
3. `education\notebooks\01_environment_setup_beginner.ipynb` 파일 확인


```text
X:\student01\
└─ workshop\                 ← 내가 만든 작업 폴더
   └─ education\             ← 제공된 교육 폴더 전체
      ├─ notebooks\          ← 교재
      ├─ data\               ← 해석 데이터
      ├─ course_utils\       ← 교재 코드가 사용하는 기능
      └─ ...
```


- **복사 범위:** Notebook뿐 아니라 데이터/공통 코드를 포함한 education 전체
- **폴더 구조:** 배포된 내부 구조 유지
- **결과 저장 위치:** education 안의 `runs`/`reports`


### 4.2 교육용 가상환경 만들기


**실행 장소: Ansys Python Manager**

1. **Create virtual environments** 탭 선택
2. **Select Python version** → 설치한 Python 3.12 선택
3. **Virtual environment name** → `omniverse-env` 입력. 다른 이름을 쓰면 이후 단계에도 같은 이름 적용
4. **Create** 선택 후 생성 완료까지 대기
5. **Manage Python environments** 탭 열기
6. **Available virtual environments** → `omniverse-env` 선택
7. 환경 경로 `X:\student01\omniverse-env` 확인
8. **Launch options → Working Directory** → `X:\student01\workshop` 입력

- **Working Directory:** 프로그램이 작업을 시작할 폴더. 드라이브를 포함한 전체 경로 입력
- **가상환경 폴더:** Manager에서 자동 생성


```text
X:\student01\
├─ omniverse-env\            ← Manager가 만든 Python/패키지 공간
└─ workshop\                 ← 프로그램이 일을 시작할 폴더
   └─ education\             ← 교재/입력 데이터/실습 결과
```


**후크 교육용 환경 확인:** `omniverse-env` 선택. 4장 전용 환경은 5.1.6절에서 별도로 준비


### 4.3 기본 패키지를 설치하고 Notebook 열기


**실행 장소: Manager의 Manage Python environments 탭**

1. `omniverse-env` 선택 → **Install Python default packages** 실행
2. **List installed packages** → 설치 목록 확인
3. **PyAnsys package → PyDPF-Core (`ansys-dpf-core`)** 설치
4. **Jupyter Notebook / Launch Jupyter Notebook** 선택. 필요한 경우 Manager에서 Notebook 설치
5. **education → notebooks → 01_environment_setup_beginner.ipynb** 열기
6. 커널 선택 창 → **Python 3 (ipykernel)** 선택

| 패키지 | 이번 교육에서 이해할 역할 |
|---|---|
| NumPy | 많은 숫자를 묶어서 계산 |
| SciPy | 과학/공학 계산 기능 |
| pandas | 행과 열로 된 표 처리 |
| Matplotlib | 그래프 그리기 |
| PyVista | 메시와 해석 결과 파일 읽기/표시 |
| scikit-learn | 머신러닝 보조 기능. 주 학습 도구는 뒤에서 설치할 PyTorch/PhysicsNeMo |

**의존성:** 패키지가 작동하는 데 필요한 다른 패키지. 실제 설치 목록은 표보다 많을 수 있으므로 Manager 버전별 목록 확인.

**교재가 보이지 않을 때**

- Working Directory/교육 파일 복사 위치 확인
- 같은 가상환경의 **Launch options → Console**에서 아래 명령 실행
- 설정한 작업 폴더에서 시작하므로 별도 폴더 이동 불필요
- Console 명령의 기준: Windows CMD


### 01장 / 셀 08 / Console 03


```bat
python -m notebook education/notebooks
```


**명령 입력:** 전체 명령 붙여넣기 → **Enter** → 선택한 가상환경의 Python으로 Notebook 실행

`(omniverse-env)`/`X:\...>`는 Console의 프롬프트이므로 입력하지 않음.


### 4.4 코드 셀을 실행하고 결과 읽기


- **Markdown 셀:** 설명 표시 / **Code 셀:** 코드 실행
- **Shift+Enter:** 선택 셀 실행 후 다음 셀로 이동
- **실행 표시:** `[ ]` 미실행 / `[*]` 실행 중 / `[1]` 등 숫자는 실행 순서
- **실행 순서:** 위에서 아래로 진행
- **Ctrl+S:** 코드와 출력 저장
- **커널 재시작:** 변수/실행 상태 초기화 → 위쪽 코드부터 다시 실행
- **패키지 설치/교체 후:** **Kernel → Restart Kernel** 선택

**첫 코드의 수정 항목: 개인 폴더/가상환경 이름 두 곳**


### 01장 / 셀 08 / 코드 조각 04


```python
PERSONAL_ROOT = Path(r'X:\student01')  # 자신의 개인 폴더
ENV_NAME = 'omniverse-env'           # Manager에서 만든 가상환경 이름
```


- **`#`:** 주석 / **`=`:** 오른쪽 값을 왼쪽 이름에 저장
- **`Path(...)`:** 경로 처리 도구
- **`r'...'`:** Windows 경로의 역슬래시를 그대로 적는 문자열 표기
- **수정 범위:** 따옴표/괄호를 유지하고 경로/환경 이름만 변경
- **`sys.executable`:** 실제 Python 실행 파일 / **`sys.prefix`:** Python 환경 위치
- **점검 방식:** 입력한 예상 환경과 실제 Notebook 커널 비교. Manager 화면을 직접 읽는 기능은 아님

참고: [Manager 패키지 설치/실행 안내](https://installer.docs.pyansys.com/version/stable/user_guide.html)


### 01장 / 셀 09 / Notebook


In [ ]:
from pathlib import Path
import sys
import os
import json

# 자신의 개인 폴더와 Manager에서 생성한 가상환경 이름으로 수정한다.
PERSONAL_ROOT = Path(r'X:\student01')
ENV_NAME = 'omniverse-env'
# PERSONAL_ROOT를 None으로 두면 '비교 경로 미설정'으로 표시한다.
EXPECTED_ENV_PREFIX = PERSONAL_ROOT / ENV_NAME if PERSONAL_ROOT is not None else None

working_dir = Path.cwd().resolve()
education_dir = next((p for base in [working_dir, *working_dir.parents]
                      for p in [base, base / 'education']
                      if (p / 'course_utils/environment.py').is_file()), None)
if education_dir is None:
    raise FileNotFoundError('교육 프로젝트 루트 또는 education 폴더에서 실행하세요.')
if str(education_dir) not in sys.path:
    sys.path.insert(0, str(education_dir))
import course_utils
from course_utils.environment import (runtime_snapshot, physics_dependencies,
                                      data_inventory, model_probe, dpf_probe, digest)

snapshot = runtime_snapshot(EXPECTED_ENV_PREFIX)
print('Python 버전:', snapshot['python_version'])
print('실제 커널 Python:', snapshot['python_executable'])
print('현재 환경:', snapshot['prefix'])
print('가상환경 여부:', snapshot['is_virtual_environment'])
environment_match_status = ('비교 경로 미설정' if EXPECTED_ENV_PREFIX is None
                            else '일치' if snapshot['manager_environment_confirmed'] else '불일치')
print('예상 가상환경 경로:', EXPECTED_ENV_PREFIX)
print('교육용 가상환경 경로 일치:', environment_match_status)
print('교육 폴더:', education_dir)
if not snapshot['is_virtual_environment'] or not snapshot['manager_environment_confirmed']:
    print('CHECK: PERSONAL_ROOT/ENV_NAME과 실제 커널 경로를 확인하세요.')


**Python 실행환경 확인**

- **`True` / `False`:** 조건 충족 / 미충족

- **정상 출력:** `가상환경 여부: True`, `교육용 가상환경 경로 일치: 일치`
- **실제 커널 Python:** 본인 가상환경의 `omniverse-env\Scripts\python.exe`
- **불일치 시:** `PERSONAL_ROOT`/`ENV_NAME` 확인 → 입력이 맞으면 Notebook의 커널 선택 점검


## 5. 필요한 기능이 실제로 작동하는지 확인하기


### 5.1 GPU와 추가 Python 패키지 준비


**GPU의 역할과 기본 개념**

- **CPU:** 전체 실행 흐름/파일 처리
- **GPU:** 많은 숫자의 병렬 계산/3D 화면 생성
- **병렬 처리:** 비슷한 계산을 여러 개 함께 수행. 케이스별 다수 노드의 학습/예측에 활용
- **CUDA:** NVIDIA GPU를 이용한 숫자 계산 기술
- **RTX:** 광선 추적/AI 가속 등을 활용하는 그래픽 기술
- **렌더링:** 형상/색/조명을 화면 이미지로 만드는 과정
- **광선 추적:** 빛의 경로를 따라 조명/그림자 계산
- **드라이버:** Windows와 GPU의 통신을 돕는 소프트웨어

| 교육 도구 | GPU가 하는 일 | 이번 교육의 준비 조건 |
|---|---|---|
| PyTorch/PhysicsNeMo | AI 모델 학습과 예측에 필요한 숫자 계산 | CUDA 계산을 지원하는 NVIDIA GPU, CUDA 기능이 포함된 PyTorch, 호환 드라이버 |
| Omniverse Kit의 RTX Renderer | 후크 형상과 결과 색을 3D 화면으로 표시 | RTX 렌더링을 지원하는 NVIDIA GPU와 그래픽 드라이버 |
| 전체 교육 실습 | 모델 계산과 3D 시각화 연결 | 위의 계산 기능과 렌더링 기능을 모두 사용할 수 있는 환경 |

**장비 선택 기준**

- PhysicsNeMo 학습: RTX 제품명에 한정되지 않으며 A100/H100 등도 공식 권장 장비에 포함
- 전체 교육: Kit RTX Renderer까지 사용하므로 **CUDA 계산과 RTX 렌더링을 모두 지원하는 GPU/드라이버 필요**

| 활용 단계 | 기대할 수 있는 장점 | 교육에서의 활용 |
|---|---|---|
| 모델 학습 | 여러 데이터의 계산을 함께 처리해 학습 시간을 줄일 수 있음 | 설정을 바꾸어 학습하고 결과를 비교하는 반복 실험 |
| 모델 예측 | 많은 지점의 값을 묶어서 계산할 때 처리 속도를 높일 수 있음 | 후크 전체의 응력/변위 분포 예측 |
| 3D 시각화 | 화면 회전/확대와 결과 갱신을 실시간으로 확인하는 데 유리 | 두께에 맞게 바뀐 형상과 응력/변위 크기 분포를 여러 방향에서 관찰 |

**성능 판단 시 주의점**

- 실제 속도 향상은 측정으로 확인
- 작은 모델/자료는 데이터 전송 비용 때문에 CPU보다 느릴 수 있음
- GPU 사용 자체가 정확도 향상을 보장하지는 않음

**실습 장비 L4**

- RT 코어/Tensor 코어/24GB 메모리 탑재
- 하드웨어상 GPU 계산/RTX 렌더링에 활용 가능
- Tensor 코어의 실제 활용 여부는 연산 방식/설정에 따라 달라짐

**단계별 확인:** 가상 장비의 GPU 할당/드라이버 → GPU 이름 표시 → CUDA 실제 계산 → Kit 실제 화면

*GPU 요구사항 조사 기준: NVIDIA 공식 문서, 2026-10-08.*


#### 5.1.1 Console에서 GPU와 Python 확인


**실행 장소: Manager에서 omniverse-env 선택 → Launch options의 Console**

설정한 Working Directory에서 시작. 아래 명령은 Notebook이 아닌 **CMD에서 하나씩 실행 → 결과 확인**.


### 01장 / 셀 11 / Console 01


```bat
nvidia-smi

python -c "import sys; print(sys.executable)"
```


| 명령 | 하는 일 | 확인할 결과 |
|---|---|---|
| `nvidia-smi` | NVIDIA GPU와 드라이버 상태 표시 | GPU 이름/Driver Version/CUDA Version |
| `python -c ...` | 짧은 Python 코드를 콘솔에서 실행 | Notebook의 Python 경로와 동일한 경로 |

- **기존 가상 장비 기준:** L4 / 드라이버 582.16 / CUDA Version 13.0
- **GPU 미표시 시:** 가상 장비의 GPU 할당/드라이버 확인
- **`nvidia-smi`의 CUDA Version:** 드라이버가 지원하는 범위
- **PyTorch의 GPU 사용 가능 여부:** 설치 후 별도 점검 필요


#### 5.1.2 설치할 PyTorch 배포본 이해


- **PyTorch 배포본:** CPU용 / CUDA 지원 배포본
- **빌드(build):** 사용할 기능을 포함해 만든 배포 프로그램
- **이번 실습:** GPU 계산을 위한 CUDA 지원 배포본 사용


#### 5.1.3 CUDA를 지원하는 PyTorch 설치


1. Notebook 저장 → **Kernel → Shut Down Kernel**로 커널 종료. 브라우저 탭만 닫으면 커널이 남을 수 있음
2. Manager → 같은 가상환경의 **Launch options → Console** 열기
3. 아래 설치 명령 전체를 한 줄로 실행. 화면의 자동 줄바꿈과 명령 구분을 혼동하지 않도록 주의

**아래 조합의 기준:** 기존 L4/드라이버 582.16 환경. 다른 장비에서는 강사가 공식 설치 안내와 드라이버 지원 범위 확인.


### 01장 / 셀 11 / Console 02


```bat
python -m pip install --upgrade "torch==2.14.1+cu130" "torchvision==0.29.1+cu130" --index-url https://download.pytorch.org/whl/cu130
```


| 명령의 일부 | 의미 |
|---|---|
| `python -m pip install` | 지금 선택한 Python 환경에 패키지 설치 |
| `--upgrade` | 지정한 조건에 맞게 설치하거나 기존 설치 갱신 |
| `torch==...` | PyTorch의 설치 버전 지정 |
| `+cu130` | CUDA 13.0을 지원하는 빌드 표시 |
| `--index-url ...` | 설치 파일을 찾을 PyTorch 공식 배포 주소 지정 |

**CUDA 라이브러리:** PyTorch 배포본에 포함된 실행 라이브러리 사용. 이번 절차에서는 CUDA Toolkit/cuDNN 별도 설치나 CUDA 코드 직접 컴파일 불필요.

**설치 완료 후: 같은 Console에서 버전/CUDA/GPU 접근 확인**


### 01장 / 셀 11 / Console 03


```bat
python -c "import torch; print('Torch:', torch.__version__); print('CUDA build:', torch.backends.cuda.is_built()); print('CUDA runtime:', torch.version.cuda); print('GPU available:', torch.cuda.is_available())"
```


| 출력 | 뜻과 다음 행동 |
|---|---|
| `No module named 'torch'` | 현재 Python에서 PyTorch를 찾지 못함. 같은 가상환경에서 설치가 완료되었는지 확인 |
| 버전에 `+cpu`, CUDA build `False` | CPU용 설치 상태. 위의 CUDA용 설치 명령과 선택한 가상환경을 다시 확인 |
| CUDA build `True`, GPU available `False` | CUDA 기능은 있지만 현재 GPU를 사용하지 못함. 드라이버/GPU 접근/실행 환경 확인 |
| 두 항목 모두 `True` | GPU 접근 가능. 뒤의 실제 연산 점검까지 확인 |

- **`True` / `False`:** 조건 충족 / 미충족
- **`import torch`:** 설치한 PyTorch 기능 불러오기
- **다음 확인:** GPU 접근 성공 여부에 이어 실제 GPU 계산도 점검


#### 5.1.4 교육용 추가 패키지 설치


**실행 장소: 같은 Console. Working Directory: workshop**

- **설치 대상:** PhysicsNeMo/Windows용 Triton 등 교육용 추가 패키지
- **Triton:** GPU용 작은 계산 프로그램을 만드는 도구
- **이름 구분:** Windows 배포명 `triton-windows` / 코드에서 불러오는 이름 `triton`
- **설치 목적:** 모델 로딩 시 미설치 경고 방지 및 실제 GPU 계산 준비
- **버전 조합:** PyTorch 2.14 ↔ Triton 3.8 계열 (`>=3.8,<3.9`)
- **requirements 파일:** 설치할 패키지 목록. `-r` 옵션으로 읽기

**입력 방법: 빈 줄로 구분된 명령을 하나씩 실행한 뒤 결과 확인**


### 01장 / 셀 11 / Console 04


```bat
python -m pip install -r education/requirements-course.txt "torch==2.14.1+cu130" "torchvision==0.29.1+cu130" --extra-index-url https://download.pytorch.org/whl/cu130

python -m pip check

python -c "import torch, physicsnemo, triton; print('torch:', torch.__version__); print('CUDA build:', torch.backends.cuda.is_built()); print('CUDA runtime:', torch.version.cuda); print('GPU available:', torch.cuda.is_available()); print('physicsnemo import: OK'); print('Triton:', triton.__version__)"
```


**목록에 직접 지정한 패키지 10개**

| 패키지 이름 | 설치 조건 | 쉬운 역할 설명 |
|---|---|---|
| `packaging` | 버전 제한 없음 | 패키지의 버전과 필요한 조건을 읽고 비교하는 도구 |
| `fastapi` | 버전 제한 없음 | 7장에서 예측 요청을 받고 결과를 돌려주는 API 구성, 8장에서 화면 연결 |
| `uvicorn` | 버전 제한 없음 | FastAPI로 만든 API 프로그램을 실행하는 서버 |
| `torch` | 명령에서 `2.14.1+cu130` 지정 | PyTorch. 숫자 계산/AI 모델/GPU 학습에 사용 |
| `torchvision` | 명령에서 `0.29.1+cu130` 지정 | PyTorch 관련 도구 모음. PhysicsNeMo가 요구하는 패키지이며, 이번 수업에서 별도 이미지 학습을 하지는 않음 |
| `nvidia-physicsnemo` | `>=2.2,<3` | 과학/공학 AI 모델을 구성하고 저장/불러오는 데 사용하는 프레임워크 |
| `triton-windows` | Windows에서 `>=3.8,<3.9` | PyTorch 2.14와 함께 사용할 GPU 계산 도구. 코드에서는 `triton`으로 불러옴 |
| `GitPython` | `==3.1.49` | Python에서 Git 저장소 정보를 다루는 도구. 모델 관련 기능의 의존성 준비에 사용 |
| `gitdb` | `==4.0.12` | GitPython이 Git 데이터를 읽는 데 사용하는 보조 도구 |
| `smmap` | `==5.0.3` | Git 데이터 파일을 읽을 때 사용하는 보조 도구 |

**설치 조건과 결과 읽기**

- `==`: 특정 버전 지정 / `>=2.2,<3`: 2.2 이상/3 미만
- 버전을 직접 지정하지 않은 항목도 다른 패키지의 의존성 조건에 영향을 받음
- torch/torchvision: 앞서 설치한 CUDA 버전을 다시 명시. 조건 충족 시 일반적으로 기존 설치 사용
- 표에 없는 의존성도 추가 설치 가능. NumPy/SciPy/pandas/Matplotlib/PyVista/scikit-learn/PyDPF/Jupyter는 Manager에서 준비
- 추가 설치 중 일부 의존성 조정 가능. 명령의 `--extra-index-url`/`+cu130` 유지
- `pip check` 정상 출력: `No broken requirements found.`
- 마지막 명령: 패키지 불러오기와 설치 버전 확인
- 설치명 `nvidia-physicsnemo` ↔ 코드의 `physicsnemo`
- 기존 PhysicsNeMo 점검 기준: 2.2.2. 실제 설치 버전/충돌 여부 기록
- 기본 패키지를 포함한 환경 재구성 목록: `requirements-environment-base.txt`

**Triton 실제 GPU 계산 확인**

같은 Console에서 준비된 점검 파일 실행 → Triton 계산과 PyTorch 계산 비교.


### 01장 / 셀 11 / Console 05


```bat
python education/preparation/check_triton_gpu.py
```


**확인 항목:** PyTorch 2.14.1+cu130 / Triton 3.8 계열 / GPU 이름 / **Triton GPU 계산: PASS**

버전/`pip check`는 설치 상태 확인용. 실제 계산 성공 여부는 GPU 점검의 PASS로 확인.

**`DLL load failed` 발생 시:** Microsoft Visual C++ 2015–2022 재배포 가능 패키지(x64)의 설치 여부 확인. 공식 Windows 지원 프로젝트의 설치 안내 참고.


#### 5.1.5 Notebook을 다시 열고 실제 계산 확인


1. 같은 가상환경에서 Notebook 다시 열기. 이미 열려 있으면 **Kernel → Restart Kernel** 실행
2. **4.4절 첫 코드부터 재실행** → 아래 두 코드 셀 차례로 실행
3. 첫 번째 셀: 패키지 이름/버전 확인. `MISSING`은 현재 커널에서 패키지를 찾지 못한 상태
4. 두 번째 셀: CUDA/의존성/모델 입출력/GPU 계산 확인

**정상 기준:** `CUDA 빌드: True` / `GPU 사용 가능: True` / GPU 정보의 `tensor_probe_passed: True`

- **텐서:** 여러 숫자를 담은 배열
- **`tensor_probe_passed`:** 작은 GPU 계산의 성공 여부
- **이번 점검:** 합성 숫자로 기본 동작 확인 → 4장 온도 모델 학습 → 5장 실제 후크 학습
- **`사용할 GPU 지정: 미설정(기본 설정 사용)`:** 기본 설정 사용 안내이며 오류가 아님
- **`visible_devices_setting: null`:** `CUDA_VISIBLE_DEVICES`를 별도 지정하지 않은 상태
- **환경변수:** 프로그램에 전달하는 설정 / JSON의 `null`: 값 없음
- **최종 확인:** GPU 접근 여부와 실제 계산 성공 여부 모두 확인

참고: [NVIDIA CUDA 프로그래밍 모델](https://docs.nvidia.com/cuda/cuda-programming-guide/01-introduction/programming-model.html), [PhysicsNeMo 시스템 요구사항](https://docs.nvidia.com/physicsnemo/latest/getting-started/system_requirements.html), [Omniverse RTX Renderer](https://docs.omniverse.nvidia.com/materials-and-rendering/latest/rtx-renderer.html), [Omniverse 기술 요구사항](https://docs.omniverse.nvidia.com/dev-guide/latest/common/technical-requirements.html), [Omniverse RTX Renderer 기능](https://docs.omniverse.nvidia.com/materials-and-rendering/latest/rtx-renderer.html), [NVIDIA L4 공식 제품 정보](https://www.nvidia.com/en-us/data-center/l4/), [NVIDIA 설명](https://docs.nvidia.com/deploy/nvidia-smi/index.html#cuda-version), [PyTorch 공식 설치 안내](https://pytorch.org/get-started/locally/), [PyTorch 유지관리자 설명](https://discuss.pytorch.org/t/pytorch-finds-cuda-despite-nvcc-not-found/166754/2), [PyTorch CUDA 배포 목록](https://download.pytorch.org/whl/cu130/torch/), [PhysicsNeMo 설치 안내](https://docs.nvidia.com/physicsnemo/latest/getting-started/installation.html), [pip 설치 옵션](https://pip.pypa.io/en/stable/cli/pip_install/), [Triton Windows 지원 프로젝트 안내](https://github.com/triton-lang/triton-windows#installation), [PyTorch 환경변수 안내](https://docs.pytorch.org/docs/2.14/cuda_environment_variables.html)


### 01장 / 셀 12 / Notebook


In [ ]:
snapshot = runtime_snapshot(EXPECTED_ENV_PREFIX)
for group, packages in snapshot['packages'].items():
    print('\n[' + group + ']')
    for name, installed in packages.items():
        print(name, ':', installed if installed is not None else 'MISSING')
if snapshot['packages']['jupyter']['notebook'] is None:
    print('CHECK: 이 Python에 notebook 배포 패키지가 없습니다. Manager의 Launch Jupyter Notebook으로 설치/실행 환경을 확인하세요.')


**패키지 실행과 GPU 연산 점검**

**점검 순서:** PyTorch CUDA/PhysicsNeMo 의존성 → 작은 Transolver의 CPU 입출력 → 별도 GPU 행렬 계산

- **`try`/`except`:** 오류 내용을 출력하고 다음 점검을 이어가기 위한 코드
- **통과 판단:** 셀 실행 종료뿐 아니라 각 항목의 결과 확인
- **`CHECK` 발생 시:** 원인/안내 확인 후 조치


### 01장 / 셀 14 / Notebook


In [ ]:
# 설치된 PyTorch의 CUDA 빌드와 GPU 접근을 구분한다.
cuda_check = {'installed': False, 'built_with_cuda': False, 'cuda_available': False}
try:
    import torch
    cuda_check = {'installed': True, 'torch_version': str(torch.__version__),
                  'torch_file': str(torch.__file__),
                  'built_with_cuda': torch.backends.cuda.is_built(),
                  'runtime_version': torch.version.cuda,
                  'cuda_available': torch.cuda.is_available(),
                  'visible_devices_setting': os.environ.get('CUDA_VISIBLE_DEVICES')}
    print('PyTorch 버전:', cuda_check['torch_version'])
    print('CUDA 빌드:', cuda_check['built_with_cuda'])
    print('PyTorch CUDA 버전:', cuda_check['runtime_version'])
    print('GPU 사용 가능:', cuda_check['cuda_available'])
    visible_devices = cuda_check['visible_devices_setting']
    print('사용할 GPU 지정:', '미설정(기본 설정 사용)' if visible_devices is None
          else repr(visible_devices))
    if not cuda_check['built_with_cuda']:
        print('CHECK: CPU용 PyTorch입니다. 5.1.3의 CUDA 빌드를 같은 환경에 설치하고 커널을 재시작하세요.')
    elif not cuda_check['cuda_available']:
        try:
            torch.cuda.init()
        except Exception as error:
            cuda_check['initialization_error'] = type(error).__name__ + ': ' + str(error)
            print('CHECK: CUDA 초기화 오류:', cuda_check['initialization_error'])
except Exception as error:
    cuda_check['error'] = type(error).__name__ + ': ' + str(error)
    print('CHECK: PyTorch 설치/import 확인:', cuda_check['error'])

try:
    dependencies = physics_dependencies()
except ImportError as error:
    dependencies = {'installed': False, 'all_declared_dependencies_satisfied': False,
                    'conflicts': [{'requirement': 'packaging', 'installed_version': None}]}
print('PhysicsNeMo 전체 선언 의존성 충족:', dependencies['all_declared_dependencies_satisfied'])
for conflict in dependencies['conflicts']:
    print('CHECK:', conflict['requirement'], '/ 현재:', conflict['installed_version'])
try:
    model_check = model_probe()
    print('점검 모델:', model_check['model_name'])
    print('모델 계산 장치:', model_check['model_device'])
    print('모델 출력 배열 모양:', model_check['stress_shape'], model_check['displacement_shape'])
    print('GPU:', model_check['gpu'])
except Exception as error:
    model_check = {'passed': False, 'error': type(error).__name__ + ': ' + str(error)}
    print('CHECK: 모델 실행환경 미완료:', model_check['error'])
print('이 점검은 합성 입력을 사용합니다. 학습/정답 평가/체크포인트 갱신은 하지 않습니다.')
print('점검 범위: 모델 생성/CPU 입출력 확인 + 별도의 GPU 텐서 연산. 실제 GPU 학습은 4/5장에서 수행합니다.')


**모델/GPU 점검 결과 확인**

- **응력 `[1, 8, 1]`:** 1케이스/8점/점마다 값 1개
- **변위 `[1, 8, 3]`:** 같은 8점/점마다 XYZ 성분 3개
- **`점검 모델`:** `physicsnemo`/`Transolver` 확인. 5장 후크 모델의 생성/입출력을 CPU에서 점검
- **결과의 의미:** 학습 전 임의 가중치의 출력. 실제 후크 예측으로 해석하지 않음
- **GPU 확인:** 모델의 CPU 입출력 점검과 별도로 GPU 행렬 계산 수행

| 항목 | 통과 조건 | 의미 |
|---|---|---|
| `available` | `True` | PyTorch에서 GPU에 접근할 수 있음 |
| `tensor_probe_passed` | `True` | GPU에서 수행한 간단한 행렬 계산이 기대한 결과와 일치 |

**전체 의존성 충족이 `False`인 경우:** 누락 패키지/버전 불일치 확인 → 문제 해결 → 다시 실행

**학습 단계와 구분:** 이번에는 기본 동작 점검. 온도 모델 GPU 학습은 4장, 후크 학습은 5장.


#### 5.1.6 4장 GNN 실습용 환경 준비


**4장 전용 환경:** Windows용 GNN 추가 패키지와 PyTorch/CUDA 조합 분리. 기존 후크 환경 유지.

Manager → 기존 교육용 환경 → Launch options → Console. 작업 위치 workshop.


### 01장 / 셀 16 / Console 01


```bat
python education/preparation/setup_models_environment.py
```


- 생성 위치: `education/.venv-models`
- 설치 조합: PyTorch 2.10.0/CUDA 12.8 + PhysicsNeMo 2.2.2 + PyG 2.7.0 + 맞는 torch-scatter
- 다운로드 수 GB / 설치 후 CPU/GPU의 작은 GNN 계산 점검
- 성공 시 **CAE 4장 모델 실습** 커널 등록
- **1장:** 설치/점검/커널 등록까지 진행. 남은 1장과 2~3장은 기존 후크 교육용 커널 유지
- **<mark class="lesson-mark" data-kind="action" title="실습 행동" style="background-color:#d6ebff;color:#242424">4장 시작</mark>:** Notebook 열기 → Kernel → Change Kernel → 해당 커널 선택
- **5장부터:** 기존 후크 교육용 커널로 복귀

다시 점검:


### 01장 / 셀 16 / Console 02


```bat
education\.venv-models\Scripts\python.exe education/preparation/check_gnn_environment.py
```


| 장 | 커널 |
|---|---|
| 1~3장 | 기존 교육용 환경 |
| 4장 | CAE 4장 모델 실습 |
| 5~8장 | 기존 후크 교육용 환경 |

GPU가 없으면 CPU만 점검. 설치 오류 해결 후 같은 명령 재실행. 다른 PC에서는 환경을 새로 설치.


### 5.2 교육 파일 구성과 VTU 읽기 점검


**실행 장소: Notebook / 아래 경로의 기준: education 폴더**

| 폴더/파일 | 들어 있는 내용 |
|---|---|
| `notebooks/` | 지금 사용하는 장별 교재 |
| `data/lhs_doe.csv` | 10개 케이스의 두께/하중 조건표 |
| `data/raw/` | 준비된 ANSYS 결과인 RST 파일 |
| `data/vtu/` | 메시와 결과를 담은 VTU 파일 |
| `course_utils/` | 코드 셀에서 불러 쓰는 공통 기능 |
| `runs/` | 데이터 정리/학습/평가 과정의 결과 |
| `reports/` | 환경과 실행 상태를 기록한 점검 파일 |
| `preparation/` | 설치/변환/실행 등을 돕는 스크립트 |
| `omniverse/` | 3D 장면과 화면을 만드는 코드 |

- **첫 번째 셀:** 10케이스의 조건/파일 확인
- **해시(hash):** 파일 내용으로 계산한 식별값. 준비 당시 값과 대조해 복사 내용의 변경 여부 확인
- **두 번째 셀:** PyVista로 `case_001.vtu` 읽기 → 노드/요소/응력/변위/단위 확인
- **점검 구분:** 해시 일치 / 실제 파일 읽기 성공. 변환 절차는 2장에서 실습

**첫 셀 중단 시:** 파일 누락/해시 불일치 해결 → 첫 셀 재실행 → `inventory` 준비 후 마지막 기록 저장


### 01장 / 셀 18 / Notebook


In [ ]:
inventory = data_inventory(education_dir)
print('case_id / thickness_mm / force_N / nodes')
for row in inventory:
    print(row['case_id'], row['thickness_mm'], row['force_N'], row['nodes'])
print('10개 RST와 VTU 파일 해시/DOE 연결 확인 완료')


**케이스 목록 확인**

| 열 이름 | 의미 |
|---|---|
| `case_id` | 케이스 이름 |
| `thickness_mm` | 두께(mm) |
| `force_N` | 하중 크기(N) |
| `nodes` | 노드 수 |

**확인 항목:** 10케이스의 이름/두께/하중/노드 수

**다음 셀:** 대표 파일 `case_001.vtu`의 배열/단위 점검


### 01장 / 셀 20 / Notebook


In [ ]:
import numpy as np
import pyvista as pv

representative_vtu = education_dir / 'data/vtu/case_001.vtu'
vtu_check = {'passed': False, 'file': str(representative_vtu)}
try:
    representative_grid = pv.read(representative_vtu)
    stress_values = np.asarray(representative_grid.point_data['von_mises_stress'])
    displacement_values = np.asarray(representative_grid.point_data['displacement'])
    units = {key: str(np.asarray(representative_grid.field_data[key]).reshape(-1)[0])
             for key in ['coordinate_unit', 'stress_unit', 'displacement_unit']}
    case_name = str(np.asarray(representative_grid.field_data['case_id']).reshape(-1)[0])
    passed = bool(case_name == 'case_001' and representative_grid.n_points > 0
                  and representative_grid.n_cells > 0
                  and stress_values.shape == (representative_grid.n_points,)
                  and displacement_values.shape == (representative_grid.n_points, 3)
                  and np.isfinite(stress_values).all() and np.isfinite(displacement_values).all()
                  and units == {'coordinate_unit':'mm', 'stress_unit':'MPa', 'displacement_unit':'mm'})
    vtu_check.update({'passed':passed, 'case_id':case_name, 'nodes':representative_grid.n_points,
                      'elements':representative_grid.n_cells, 'stress_shape':list(stress_values.shape),
                      'displacement_shape':list(displacement_values.shape), 'units':units,
                      'max_stress_mpa':float(stress_values.max())})
    print('VTU:', case_name, representative_grid.n_points, 'nodes /', representative_grid.n_cells, 'elements')
    print('응력 / 변위 형태:', stress_values.shape, '/', displacement_values.shape)
    print('단위:', units)
    print('대표 VTU 실제 읽기 점검:', passed)
except Exception as error:
    vtu_check['error'] = type(error).__name__ + ': ' + str(error)
    print('CHECK: 대표 VTU 읽기/필드/단위 확인:', vtu_check['error'])


**VTU 점검 항목과 출력 해석**

| 출력 | 의미 |
|---|---|
| `13668 nodes / 8024 elements` | 계산용 점 13,668개와 요소 8,024개가 들어 있음 |
| 응력 배열 모양 `(13668,)` | 각 노드에 응력 값 하나가 있음. 쉼표는 Python 배열 모양 표기 |
| 변위 배열 모양 `(13668, 3)` | 각 노드에 X/Y/Z 방향 변위 세 개가 있음 |
| `coordinate_unit: mm` | 좌표 단위 |
| `stress_unit: MPa` | 응력 단위 |
| `displacement_unit: mm` | 변위 단위 |
| 대표 VTU 실제 읽기 점검 `True` | 필요한 데이터와 단위를 읽는 점검 통과 |

- **`np.asarray`:** 데이터를 숫자 배열로 변환
- **`shape`:** 배열의 축별 크기
- **`np.isfinite`:** NaN/무한대 여부 확인
- **노드별 대응:** 응력 값 1개/변위 XYZ 성분 3개

**실제 단위 출력:** `{'coordinate_unit': 'mm', 'stress_unit': 'MPa', 'displacement_unit': 'mm'}`


### 5.3 PyDPF/DPF 서버 실행환경 점검


**실행 장소: Notebook / 필요 프로그램: ANSYS 2025 R2 DPF 서버**

- **PyDPF-Core:** Python에서 해석 결과 읽기를 요청하는 클라이언트 도구
- **DPF 서버:** 요청을 받아 RST를 처리하는 프로그램. ANSYS 2025 R2에 포함된 서버 사용


```text
Notebook의 Python → PyDPF로 읽기 요청 → DPF 서버 → RST 파일 읽기
```


- **실행 위치:** 클라이언트와 서버를 같은 장비에서 실행 가능
- **설치 구분:** PyDPF 설치만으로 DPF 서버가 설치되지는 않음
- **ANSYS 미설치 시:** 미완료로 기록 → 설치 후 재점검

**ANSYS 경로 설정**

- `ANSYS_DIR`: 실제 **v252 폴더**의 위치
- `AWP_ROOT252`가 있으면 해당 경로 사용, 없으면 기본 설치 경로 사용
- 다른 위치에 설치한 경우: **다음 실행 코드 셀의 첫 줄 전체**를 아래 `Path(...)` 형태로 교체
- 아래는 설명용 예시. 예시만 수정하면 실행 셀에는 반영되지 않음


### 01장 / 셀 22 / 코드 조각 02


```python
ANSYS_DIR = Path(r'D:\실제설치폴더\ANSYS Inc\v252')
```


**점검 순서:** 서버 시작 → `case_001`의 기존 변위 읽기 → 서버 종료

새 해석 없이 준비된 RST에서 기존 해석 결과를 읽는 기능 확인.

참고: [PyDPF 서버 실행 API](https://dpf.docs.pyansys.com/version/stable/api/ansys/dpf/core/server/index.html)


### 01장 / 셀 23 / Notebook


In [ ]:
ANSYS_DIR = Path(os.environ.get('AWP_ROOT252', 'C:/Program Files/ANSYS Inc/v252'))
if ANSYS_DIR.is_dir():
    os.environ['AWP_ROOT252'] = str(ANSYS_DIR)
    try:
        dpf_check = dpf_probe(ANSYS_DIR, education_dir / 'data/raw/case_001.rst')
        print('DPF 실제 읽기 점검:', json.dumps(dpf_check, ensure_ascii=False, indent=2))
    except Exception as error:
        dpf_check = {'passed': False, 'error': type(error).__name__ + ': ' + str(error)}
        print('CHECK: DPF 경로/클라이언트/서버 연결 확인:', dpf_check['error'])
else:
    dpf_check = {'passed': False, 'error': 'ANSYS 2025 R2 installation directory not found'}
    print('CHECK: ANSYS_DIR를 실제 v252 설치 폴더로 바꾸세요.')


**DPF 점검 결과 확인**

- **`passed: true`:** 서버 시작/RST 변위 읽기 성공
- **JSON 표기:** 소문자 `true`/`false`
- **`false`/`CHECK` 발생 시:** 오류 내용/`ANSYS_DIR` 확인
- **ANSYS 미설치 시:** 설치 후 다시 실행
- **점검 범위:** 기존 결과 읽기. 새 해석 수행 여부와 구분


### 5.4 Omniverse Kit 준비하기


**Kit:** CAE 자료/AI 결과를 3D 화면에 연결할 프로그램의 실행 도구. Python 패키지와 별도 준비.

**준비 항목**

- Git for Windows/인터넷: 공식 파일 다운로드
- RTX 렌더링 지원 GPU/그래픽 드라이버: 3D 화면 표시
- 가상 장비 드라이버: 해당 서비스의 안내에 따라 준비

CUDA 계산 성공과 실제 렌더링 성공은 각각 확인 필요.

**실행 장소: 같은 가상환경 Console / 작업 위치: workshop**

**설치 전 Git 확인**


### 01장 / 셀 25 / Console 01


```bat
git --version
```


버전 출력 → 설치 진행. 명령이 없으면 Git for Windows 설치 → Console 새로 열기 → 재확인. 설치 권한이 없으면 강사에게 요청.

**신규 설치 명령**


### 01장 / 셀 25 / Console 02


```bat
python education/preparation/install_omniverse_kit.py --install-dir "X:\ovedu"
```


- **`--install-dir`:** 새 폴더 또는 빈 폴더 지정. `X:\ovedu`는 짧은 설치 경로 예시
- **설치 내용:** Kit 110.2 계열의 공식 저장소 버전을 고정하고 필요한 확장 준비
- **확장:** Kit에 기능을 더하는 구성 요소
- **설치 중:** 약관/다운로드 안내 확인
- **실패 시:** 오류를 보관하고 강사와 확인. 같은 설치 폴더에서 재시도 가능
- **설치 경로 저장:** `education/config/kit_runtime.json`
- **JSON:** 항목 이름과 값을 담는 텍스트 형식
- **커널 재시작 후:** 위쪽 코드부터 다시 실행해 저장 경로 읽기

**호환 Kit가 이미 있는 경우: 실행 파일 등록**


### 01장 / 셀 25 / Console 03


```bat
python education/preparation/install_omniverse_kit.py --register-existing "실제 SDK 경로/kit.exe"
```


- **등록 경로:** 실제 `kit.exe`의 전체 경로. `kit.bat`와 구분
- **Template 환경의 일반적인 위치:** `_build\windows-x86_64\release\kit\kit.exe`
- **SDK:** 개발/실행 도구 묶음
- **다음 셀:** 위치/기본 실행 명령 출력
- **Kit 창 열기:** 출력된 명령을 Console에서 실행

참고: [Kit 요구사항](https://docs.omniverse.nvidia.com/kit/docs/kit-app-template/latest/docs/intro.html), [NVIDIA 드라이버 안내](https://docs.omniverse.nvidia.com/dev-guide/latest/common/technical-requirements.html), [공식 앱 빌드 안내](https://github.com/NVIDIA-Omniverse/kit-app-template/blob/main/readme-assets/additional-docs/kit_app_template_tooling_guide.md)


### 01장 / 셀 26 / Notebook


In [ ]:
from preparation.launch_kit import configured_kit
KIT_EXE = configured_kit()  # 설치/등록 명령이 저장한 경로. 변경 시 --register-existing으로 재등록한다.
kit_check = {'kit_executable': str(KIT_EXE), 'binary_exists': KIT_EXE.is_file(),
             'launcher_exists': (education_dir / 'preparation/launch_kit.py').is_file(),
             'startup_checked': False}
print('Kit 점검:', kit_check)
print('기본 실행 명령: python education/preparation/launch_kit_basic_revised.py --kit-exe "' + str(KIT_EXE) + '"')


**Kit 실행 파일 경로 확인**

- **`binary_exists: True`:** 지정 위치에 `kit.exe` 존재
- **다음 작업:** 출력된 기본 명령을 Console에서 실행 → 창/장면 확인
- **`binary_exists: False`:** 5.4절 `--register-existing`으로 실제 `kit.exe` 재등록 → 위 경로 확인 셀 재실행 → 새 기본 명령 사용


#### Kit 실행 및 기본 장면 확인


1. 위 셀의 기본 명령에서 `python`부터 전체 복사
2. 같은 가상환경의 Console에서 실행. 작업 위치는 workshop
3. **Chapter 1 - Kit Environment Check** 창의 파란 큐브 확인
4. 초기 렌더링 준비 후 화면/오류 확인
5. `education\reports\kit_basic_startup_revised.json` 생성 후 창 닫기. 너무 일찍 종료한 경우 재실행
6. 아래 코드의 `KIT_BASIC_VIEW_CONFIRMED`/`KIT_BASIC_NOTES`에 실제 관찰 결과 입력

**큐브의 역할:** AI/API 없이 기본 장면 실행 여부 점검. 후크 연결은 8장에서 진행.

| 내가 확인한 상태 | 입력할 값 |
|---|---|
| 아직 실행하지 않았거나 화면을 확인하지 못함 | `None` 그대로 유지 |
| 창에 파란 큐브가 정상으로 보임 | `True` |
| 직접 실행했으나 검은 화면/오류 등 문제가 있음 | `False`와 관찰 내용 |

**정상 관찰 시 입력 예시:** `True`는 대문자로 시작하며 따옴표 없이 입력


### 01장 / 셀 28 / 코드 조각 01


```python
KIT_BASIC_VIEW_CONFIRMED = True
KIT_BASIC_NOTES = '파란 큐브가 보이고 창을 정상 종료함'
```


- **`None`:** 아직 확인하지 않은 상태
- **통과 조건:** 실제 관찰과 장면 기록/캡처 모두 필요. 관찰 항목에 `True`만 입력해도 전체 통과하는 것은 아님
- **로그(log)/캡처 위치:** `runs/environment/kit_basic_revised/`
- **오류 시 보관:** 화면과 `stdout.log`/`stderr.log`/`kit.log`
- **첫 실행의 셰이더 컴파일:** GPU 화면을 준비하는 과정
- **`Waiting for RtPso ... compilation`:** 준비 대기 메시지
- **자동 점검:** 명령 뒤에 `--verify` 추가 → 렌더링/캡처 후 자동 종료. 최대 대기 시간 10분
- **기본 실습:** 실제 창을 직접 보고 관찰 결과 기록


### 01장 / 셀 29 / Notebook


In [ ]:
# 기본 Kit 창을 실제 확인한 뒤 True/False 입력. 아직 확인하지 않았다면 None 유지.
KIT_BASIC_VIEW_CONFIRMED = None
KIT_BASIC_NOTES = ''

startup_path = education_dir / 'reports/kit_basic_startup_revised.json'
startup = json.loads(startup_path.read_text(encoding='utf-8')) if startup_path.is_file() else {}
reported_exe = startup.get('kit_executable')
same_executable = bool(reported_exe and Path(reported_exe).resolve() == KIT_EXE.resolve())
capture = education_dir / startup.get('viewport_capture', 'runs/environment/kit_basic_revised/viewport.png')
kit_basic_ready = bool(kit_check['binary_exists'] and same_executable
                       and startup.get('scene_initialized') and startup.get('render_capture_written')
                       and capture.is_file() and KIT_BASIC_VIEW_CONFIRMED is True)
kit_check.update({'startup_checked': bool(startup), 'startup_report':startup,
                  'same_executable':same_executable, 'user_view_confirmed':KIT_BASIC_VIEW_CONFIRMED,
                  'user_notes':KIT_BASIC_NOTES, 'basic_startup_ready':kit_basic_ready})
print('기본 Kit 화면 확인:', '미확인' if KIT_BASIC_VIEW_CONFIRMED is None else KIT_BASIC_VIEW_CONFIRMED)
print('기본 Kit 준비 완료:', kit_basic_ready)
if not kit_basic_ready:
    print('CHECK: 기본 실행 명령, 장면/캡처 기록, 실제 큐브 화면 확인 결과를 점검하세요.')


**Kit 점검 결과 확인**

- **`기본 Kit 준비 완료: True`:** 실행 파일/장면 기록/캡처/직접 화면 확인의 조건 충족
- **`False`인 경우:** 미충족 항목 확인
- **이번 점검:** 기본 장면 실행
- **이후 연결:** 7장 API 서버 → 8장 Omniverse 화면


## 6. 실행환경 점검 결과 저장


**저장 대상:** Python/패키지/GPU/데이터/DPF/Kit의 점검 결과

커널을 재시작했다면 앞 코드부터 재실행해야 함.

**<mark class="lesson-mark" data-kind="action" title="실습 행동" style="background-color:#d6ebff;color:#242424">기존 후크 교육용 환경 준비 완료</mark>의 확인 항목**

- 예상 가상환경과 실제 Notebook 커널 일치
- 패키지 설치/의존성 버전 조건 충족
- GPU 접근/실제 소규모 계산 성공
- VTU/ANSYS 결과 읽기 성공
- Kit 실행 기록/캡처 확보 및 파란 큐브 관찰 완료

**4장 전용 환경:** <mark class="lesson-mark" data-kind="caution" title="주의사항" style="background-color:#f8dce4;color:#242424">위 결과와 별개</mark>. 5.1.6절의 설치 점검 성공/커널 등록 확인

**미완료 항목:** `False`/`CHECK`의 원인 확인 후 기록. 모델 학습/API/화면 연계는 이후 장에서 점검.

**결과 저장 위치:** `reports/environment_lesson_check_beginner.json`

**Notebook 저장:** **Ctrl+S**


### 01장 / 셀 32 / Notebook


In [ ]:
required_packages_present = all(value is not None for group in snapshot['packages'].values()
                                for value in group.values())
gpu_check = model_check.get('gpu', {})
gpu_ready = bool(gpu_check.get('available') and gpu_check.get('tensor_probe_passed'))
classroom_environment_ready = bool(
    snapshot['python_312'] and snapshot['is_virtual_environment']
    and snapshot['manager_environment_confirmed'] and required_packages_present
    and dependencies['all_declared_dependencies_satisfied'] and model_check['passed'] and gpu_ready
    and dpf_check['passed'] and vtu_check['passed'] and kit_basic_ready)
lesson_report = {'snapshot': snapshot, 'dependencies': dependencies, 'model_probe': model_check,
                 'dpf_probe': dpf_check, 'kit_check': kit_check, 'inventory': inventory, 'vtu_read_probe': vtu_check,
                 'classroom_environment_ready': classroom_environment_ready, 'gpu_required': True, 'gpu_ready': gpu_ready,
                 'cuda_build_check': cuda_check,
                 'readiness_scope': 'Package/kernel/GPU tensor/DPF/actual VTU read/basic Kit scene and user view confirmation; inference API validation in module 7; Hook viewer validation in module 8',
                 'manager_gui_installation_executed_by_notebook': False,
                 'new_analysis_or_training_performed': False}
report_path = education_dir / 'reports/environment_lesson_check_beginner.json'
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(lesson_report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('GPU 교육 환경 준비:', gpu_ready)
print('기존 후크 교육용 환경 준비 완료:', classroom_environment_ready)
print('점검 기록:', report_path)
if not classroom_environment_ready:
    print('CHECK 항목을 정리하세요. 실행 성공만으로 전체 설치 완료를 판정하지 않습니다.')


**점검 기록 확인**

- **`점검 기록:` 출력:** JSON 파일 저장 경로 확인
- **결과가 만들어진 점검:** 일부 `False`여도 기록 저장 가능. `CHECK` 항목 확인
- **오류로 셀이 중단된 경우:** 원인 해결 → 해당 셀부터 재실행 → 마지막 기록 저장
- **마지막 작업:** Notebook 저장

**다음 장:** [2장. CAE 학습 데이터 준비와 ANSYS 결과 변환](02_rst_to_vtu_beginner.ipynb) — DOE 조건 선정 → Workbench에서 해석 과정 관리 → ANSYS 결과 확보 → PyDPF 조회/VTU 변환

**수업 범위:** 데이터 제작 과정 이해 / 실제 실습은 준비된 RST 사용
